In [4]:
from dataclasses import dataclass

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int 

In [1]:
import json

def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [2]:
test_bytes = json.dumps({
    'PULocationID': 186,
    'DOLocationID': 79,
    'trip_distance': 1.72,
    'total_amount': 17.31,
    'tpep_pickup_datetime': 1730429702000
}).encode('utf-8')



In [5]:
ride_deserializer(test_bytes)
# Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72,
#      total_amount=17.31, tpep_pickup_datetime=1730429702000)

Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72, total_amount=17.31, tpep_pickup_datetime=1730429702000)

# Consumer
A **KafkaConsumer** works the same way as a producers, but in reverse: it's a continuous loop that pulls messages one at a time (or in small internal batches, but exposed to code one message per loop iteration), not "give me everything you have right now."

**Consumers** are long-running, not scheduled, it will happily keep running past 1000 if the producer sends more later — it has no built-in concept of "the batch is 1000 rides," it just knows "keep giving me whatever's next in the topic."

**Consumer workflow**
1) The consumer connects and starts a long-running loop.
2) Each iteration, it pulls the next unread message from the topic (tracked via its offset/group_id)— bytes off the broker's disk.
3) Those bytes are automatically deserialized (JSON → dict → Ride object) by the function registered



In [11]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

/tmp/ipykernel_53599/2631740412.py:6: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


# group_id
Every message in a partition has a sequential offset (0, 1, 2, 3...), and the broker tracks, per consumer group, "how far has this group read?" group_id='rides-console' is the name of that bookmark.
After your consumer reads and processes messages, it periodically commits its current offset back to the broker — essentially telling Redpanda "**the group named rides-console has successfully read up through offset N**." The broker stores this bookmark. It's not stored in your Python script or on your laptop — it lives on the broker itself, tied to that group name string.

**Example walkthrough:**

Say your rides topic (single partition, for simplicity) has offsets 0 through 999 — your 1000 produced rides.

1) You run the consumer script for the first time with group_id='rides-console'. No bookmark exists yet for this group name.
2) It reads messages, processes them (prints them, whatever), and commits offsets as it goes. In the below loop after offset=10 it closes the consumer. 
3) The broker now has stored: "**group rides-console is at offset 10.**"
4) You run the exact same script again, same group_id. This time, instead of starting over, it asks the broker "where was I?", gets back "offset 10," and resumes from message 10 — you see rides 11 through 20, not rides 0-10 again.

If you instead ran the script with **group_id='some-other-name'**, the broker has never heard of that group before, so it has no bookmark for it — it would **start from scratch**, re-reading from the beginning (subject to auto_offset_reset, next section), completely independently of rides-console's progress. This is also how you'd run two genuinely different consumers against the same topic simultaneously — a "console printer" and a "postgres writer" — each with its own group_id, each tracking its own independent progress through the same messages.



# auto_offset_reset
This setting only matters in one specific situation: a group_id with no committed offset stored yet (either truly brand new, or its bookmark expired/was deleted). It answers: "since I have no bookmark to resume from, where do I start?"

Two options:

1) **'earliest'** — **start from offset 0**, the very oldest message still retained on disk. You get the full history.
2) **'latest'** — start from whatever **offset is newest right now** — meaning you only see messages produced after your consumer connects. Anything already sitting on the broker gets skipped.

Say it's a brand new **group_id='rides-live-dashboard'** that has never run before, and all 1000 rides are already sitting on the broker (you ran the producer earlier, in a previous cell).

1) With auto_offset_reset='earliest': this consumer starts at offset 0 and reads all 1000 existing rides, top to bottom.
2) With auto_offset_reset='latest': this consumer jumps straight to "the end" — offset 1000 (i.e., "wait for whatever comes next"). It sees zero of the 1000 already-produced rides. It'll just sit there, blocked, until a new ride gets produced after it connected.

Scenario:
1) A New group_id with auto_offset_reset='earliest'	will Reads everything from offset 0
2) A New group_id with auto_offset_reset='latest' will Reads nothing old; waits for new messages only
3) An existing group_id with a saved bookmark. The auto_offset_reset is ignored entirely — resumes from the saved offset, regardless of which setting you passed

Note: if you run your consumer once, stop it, then change the code and rerun it with the same group_id, changing auto_offset_reset from 'earliest' to 'latest' won't do anything — the broker already has a bookmark for rides-console, so it just resumes from there.

To force a genuine "start over from the beginning," you'd need to either pick a brand-new group_id, or explicitly reset/delete the existing group's offset via the broker's admin tools (rpk group commands, in Redpanda's case).

In [12]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...
Received: PU=238, DO=24, distance=0.6, amount=$9.12, pickup=2025-11-01 00:03:18
Received: PU=236, DO=147, distance=4.3, amount=$29.20, pickup=2025-11-01 00:19:55
Received: PU=231, DO=137, distance=3.0, amount=$32.75, pickup=2025-11-01 00:45:55
Received: PU=237, DO=237, distance=0.69, amount=$11.50, pickup=2025-11-01 00:11:12
Received: PU=132, DO=265, distance=15.47, amount=$106.63, pickup=2025-11-01 00:05:21
Received: PU=79, DO=125, distance=1.29, amount=$22.26, pickup=2025-11-01 00:09:20
Received: PU=158, DO=79, distance=1.66, amount=$32.34, pickup=2025-11-01 00:38:59
Received: PU=79, DO=90, distance=1.25, amount=$22.25, pickup=2025-11-01 00:55:33
Received: PU=142, DO=249, distance=2.68, amount=$48.68, pickup=2025-11-01 00:16:24
Received: PU=4, DO=48, distance=3.16, amount=$33.15, pickup=2025-11-01 00:53:29

... received 10 messages so far (stopping after 10 for demo)


### Below the outputs are produced when the above code is ran twice. The output shows diffrent rows of the taxi data set. Because the consumer is ran twice with the same group_id, it is bookmarked and after each run the next rows are showed.

In [ ]:
Listening to rides...
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-11-01 00:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-11-01 00:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-11-01 00:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-11-01 00:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-11-01 00:13:25
Received: PU=142, DO=237, distance=2.28, amount=$24.94, pickup=2025-11-01 00:49:07
Received: PU=163, DO=238, distance=2.7, amount=$25.62, pickup=2025-11-01 00:07:19
Received: PU=138, DO=261, distance=12.87, amount=$86.14, pickup=2025-11-01 00:00:00
Received: PU=138, DO=37, distance=8.4, amount=$48.65, pickup=2025-11-01 00:18:50
Received: PU=90, DO=100, distance=0.85, amount=$16.45, pickup=2025-11-01 00:21:11

... received 10 messages so far (stopping after 10 for demo)

Listening to rides...
Received: PU=142, DO=170, distance=3.01, amount=$25.85, pickup=2025-11-01 00:07:31
Received: PU=237, DO=144, distance=3.82, amount=$57.54, pickup=2025-11-01 00:46:52
Received: PU=162, DO=161, distance=0.89, amount=$12.95, pickup=2025-11-01 00:56:59
Received: PU=234, DO=162, distance=2.28, amount=$38.68, pickup=2025-11-01 00:10:43
Received: PU=158, DO=88, distance=3.3, amount=$44.00, pickup=2025-11-01 00:00:03
Received: PU=88, DO=148, distance=1.5, amount=$19.55, pickup=2025-11-01 00:43:53
Received: PU=148, DO=236, distance=4.7, amount=$47.65, pickup=2025-11-01 00:58:02
Received: PU=87, DO=255, distance=5.61, amount=$38.85, pickup=2025-11-01 00:52:48
Received: PU=231, DO=43, distance=3.9, amount=$46.55, pickup=2025-11-01 00:05:53
Received: PU=141, DO=262, distance=1.14, amount=$14.90, pickup=2025-11-01 00:13:44

... received 10 messages so far (stopping after 10 for demo)